# Week 4: Window functions

In [1]:
import duckdb

con = duckdb.connect("../sql/taxi_project")

In [5]:
con.sql("""
    SELECT * FROM zone_lookup WHERE LocationID = 1;
""")

┌────────────┬─────────┬────────────────┬──────────────┐
│ LocationID │ Borough │      Zone      │ service_zone │
│   int64    │ varchar │    varchar     │   varchar    │
├────────────┼─────────┼────────────────┼──────────────┤
│          1 │ EWR     │ Newark Airport │ EWR          │
└────────────┴─────────┴────────────────┴──────────────┘

In [9]:
con.sql("""
    SELECT * FROM taxi_clean WHERE pickup_datetime = '2025-01-01 00:18:38';
""")

┌─────────────────────┬─────────────────────┬─────────────────┬───────────────┬────────────────┬────────────────┬─────────────┬────────────┐
│   pickup_datetime   │  dropoff_datetime   │ passenger_count │ trip_distance │ pu_location_id │ do_location_id │ fare_amount │ tip_amount │
│      timestamp      │      timestamp      │      int64      │    double     │     int32      │     int32      │   double    │   double   │
├─────────────────────┼─────────────────────┼─────────────────┼───────────────┼────────────────┼────────────────┼─────────────┼────────────┤
│ 2025-01-01 00:18:38 │ 2025-01-01 00:26:59 │               1 │           1.6 │            229 │            237 │        10.0 │        3.0 │
│ 2025-01-01 00:18:38 │ 2025-01-01 00:23:29 │               1 │          0.85 │            239 │            142 │         7.2 │       3.05 │
└─────────────────────┴─────────────────────┴─────────────────┴───────────────┴────────────────┴────────────────┴─────────────┴────────────┘

*Rank pickup zones by daily revenue*

In [28]:
con.sql("""
    WITH revenue_by_pu_location AS (
            SELECT  
                    DATE(pickup_datetime) AS Date,
                    pu_location_id AS location_id,
                    ROUND(SUM(fare_amount), 2) AS revenue
            FROM taxi_clean
            GROUP BY Date, location_id
        ),
        ranked_location AS (
            SELECT 
                    Date,
                    revenue,
                    location_id,
                    RANK() OVER (
                            PARTITION BY Date
                            ORDER BY revenue DESC
                            ) AS rank
            FROM revenue_by_pu_location
        ) 
        SELECT 
                r.Date,
                z.Zone,
                r.location_id,
                r.revenue,
                r.rank
        FROM ranked_location AS r
        INNER JOIN zone_lookup AS z
        ON r.location_id = z.LocationID
        ORDER BY r.rank, r.Date LIMIT 20;
""")

┌────────────┬─────────────┬─────────────┬───────────┬───────┐
│    Date    │    Zone     │ location_id │  revenue  │ rank  │
│    date    │   varchar   │    int32    │  double   │ int64 │
├────────────┼─────────────┼─────────────┼───────────┼───────┤
│ 2024-12-31 │ JFK Airport │         132 │      73.7 │     1 │
│ 2025-01-01 │ JFK Airport │         132 │ 307853.61 │     1 │
│ 2025-01-02 │ JFK Airport │         132 │ 336847.75 │     1 │
│ 2025-01-03 │ JFK Airport │         132 │ 290531.02 │     1 │
│ 2025-01-04 │ JFK Airport │         132 │ 302722.87 │     1 │
│ 2025-01-05 │ JFK Airport │         132 │ 345352.74 │     1 │
│ 2025-01-06 │ JFK Airport │         132 │ 324399.98 │     1 │
│ 2025-01-07 │ JFK Airport │         132 │ 279517.25 │     1 │
│ 2025-01-08 │ JFK Airport │         132 │ 251281.12 │     1 │
│ 2025-01-09 │ JFK Airport │         132 │ 285841.63 │     1 │
│ 2025-01-10 │ JFK Airport │         132 │ 299390.17 │     1 │
│ 2025-01-11 │ JFK Airport │         132 │ 268385.79 │ 

*Find top 3 zones per borough*

In [47]:
con.sql("""
    WITH revenue AS (
            SELECT
                ROUND(SUM(fare_amount), 2) AS revenue,
                pu_location_id AS location_id
            FROM taxi_clean
            GROUP BY location_id
        ),
        ranked_zones AS (
            SELECT 
                r.location_id,
                r.revenue,
                z.Zone,
                z.Borough,
                ROW_NUMBER() OVER(PARTITION BY z.Borough
                            ORDER BY r.revenue DESC
                        ) AS rank
            FROM revenue AS r
            INNER JOIN zone_lookup as z
            ON r.location_id = z.LocationID
        )
        SELECT 
                rank,
                Zone,
                Borough,
                revenue
        FROM ranked_zones
        WHERE rank <=  3
        ORDER BY Borough, rank;
""")

┌───────┬──────────────────────────────────┬───────────────┬────────────┐
│ rank  │               Zone               │    Borough    │  revenue   │
│ int64 │             varchar              │    varchar    │   double   │
├───────┼──────────────────────────────────┼───────────────┼────────────┤
│     1 │ Co-Op City                       │ Bronx         │    14821.6 │
│     2 │ East Concourse/Concourse Village │ Bronx         │    11881.8 │
│     3 │ Williamsbridge/Olinville         │ Bronx         │    11744.4 │
│     1 │ East New York                    │ Brooklyn      │   41405.08 │
│     2 │ Crown Heights North              │ Brooklyn      │   32672.96 │
│     3 │ Downtown Brooklyn/MetroTech      │ Brooklyn      │   29001.21 │
│     1 │ Newark Airport                   │ EWR           │    5551.42 │
│     1 │ Midtown Center                   │ Manhattan     │ 2203934.01 │
│     2 │ Times Sq/Theatre District        │ Manhattan     │ 1874488.38 │
│     3 │ Upper East Side South       

*Calculate 7-day moving average of trips*

In [ ]:
con.sql("""
    
""")

*Calculate day-over-day revenue change*

In [ ]:
con.sql("""

""")

*Calculate percent of total revenue by zone*

In [ ]:
con.sql("""

""")

*Find each zone's best revenue day*

In [ ]:
con.sql("""

""")

*Compare each trip fare to zone average*

In [ ]:
con.sql("""

""")

*Find unusually high fares within each zone*

In [ ]:
con.sql("""

""")

*Calculate cumulative monthly revenue*

In [ ]:
con.sql("""

""")

*Rank routes by tip percentage*

In [ ]:
con.sql("""

""")